# Initialize Store & Generate Tile List

Run this notebook **once** before any tile processing.

1. Reads `config.txt` and shows dataset parameters
2. Builds a GeoDataFrame of all 10°×10° tiles, intersects with Natural Earth land (via `geodatasets`), and saves `tile_list.geojson` — committed to the repo and not recomputed in CI
3. Creates the empty Icechunk/Zarr v3 store on Azure Blob Storage

After this, trigger the **Process All Tiles** GitHub Actions workflow to fill the store.

In [ ]:
import os
import sys
from pathlib import Path

import dask.array as da
import geodatasets
import geopandas as gpd
import icechunk
import numpy as np
import xarray as xr
from shapely.geometry import box

sys.path.insert(0, str(Path.cwd().parent))
from utils import load_config

cfg = load_config()
YEARS           = cfg["YEARS"]
RESOLUTION      = cfg["RESOLUTION"]
TILE_SIZE_DEG   = cfg["TILE_SIZE_DEG"]
PIXELS_PER_TILE = cfg["PIXELS_PER_TILE"]
TILE_ROWS       = cfg["TILE_ROWS"]
TILE_COLS       = cfg["TILE_COLS"]
FILL_VALUE      = cfg["FILL_VALUE"]

print("Config:", cfg)

## 1. Build tile list and save as GeoJSON

Creates a GeoDataFrame of all tiles, intersects with Natural Earth land polygons loaded via `geodatasets`, and saves `tile_list.geojson`.
Commit this file to the repo — CI reads it directly without needing to recompute.

In [ ]:
# Build GeoDataFrame of all tiles
tiles = []
for row in range(TILE_ROWS):
    for col in range(TILE_COLS):
        lat_min = -90 + row * TILE_SIZE_DEG
        lon_min = -180 + col * TILE_SIZE_DEG
        tiles.append({
            "row": row, "col": col,
            "geometry": box(lon_min, lat_min, lon_min + TILE_SIZE_DEG, lat_min + TILE_SIZE_DEG),
        })

all_tiles_gdf = gpd.GeoDataFrame(tiles, crs="EPSG:4326")

# Natural Earth land polygons via geodatasets
land_gdf = gpd.read_file(geodatasets.get_url("naturalearth land"))

# Keep tiles that intersect land
land_mask = all_tiles_gdf.intersects(land_gdf.union_all())
tile_list_gdf = all_tiles_gdf[land_mask].copy().reset_index(drop=True)

tile_list_path = Path.cwd().parent / "tile_list.geojson"
tile_list_gdf.to_file(tile_list_path, driver="GeoJSON")
print(f"{len(tile_list_gdf)} land tiles written to {tile_list_path}")
tile_list_gdf.head()

## 2. Initialize the Icechunk store

Set your Azure credentials before running. This creates only metadata and coordinates — no data chunks until tile runners fill them.

```python
os.environ["AZURE_STORAGE_ACCOUNT"] = "..."
os.environ["AZURE_STORAGE_SAS_TOKEN"] = "..."
os.environ["AZURE_CONTAINER"] = "..."
os.environ["ICECHUNK_PREFIX"] = "modis-lst-demo"
```

In [ ]:
n_lat  = TILE_ROWS * PIXELS_PER_TILE
n_lon  = TILE_COLS * PIXELS_PER_TILE
shape  = (len(YEARS), n_lat, n_lon)
chunks = (1, PIXELS_PER_TILE, PIXELS_PER_TILE)

lats = np.arange(90, -90, -RESOLUTION) - RESOLUTION / 2
lons = np.arange(-180, 180,  RESOLUTION) + RESOLUTION / 2

var_attrs = {
    "scale_factor": np.float32(0.02),
    "add_offset": np.float32(0.0),
    "_FillValue": FILL_VALUE,
    "valid_range": [7500, 65535],
    "units": "K",
    "grid_mapping": "spatial_ref",
}

ds = xr.Dataset(
    {
        "avg_daytime_lst": xr.DataArray(
            da.full(shape, np.uint16(FILL_VALUE), dtype=np.uint16, chunks=chunks),
            dims=["year", "latitude", "longitude"],
            attrs={**var_attrs, "long_name": "Annual mean daytime land surface temperature"},
        ),
        "max_daytime_lst": xr.DataArray(
            da.full(shape, np.uint16(FILL_VALUE), dtype=np.uint16, chunks=chunks),
            dims=["year", "latitude", "longitude"],
            attrs={**var_attrs, "long_name": "Annual maximum daytime land surface temperature"},
        ),
    },
    coords={"year": np.array(YEARS), "latitude": lats, "longitude": lons},
)
ds.attrs = {
    "title": "MODIS MOD11A2 Annual Daytime Land Surface Temperature",
    "source": "MODIS Terra MOD11A2 Version 6.1 via Microsoft Planetary Computer",
    "Conventions": "CF-1.8",
}
print(ds)

In [ ]:
storage = icechunk.azure_storage(
    account=os.environ["AZURE_STORAGE_ACCOUNT"],
    container=os.environ["AZURE_CONTAINER"],
    prefix=os.environ["ICECHUNK_PREFIX"],
    sas_token=os.environ["AZURE_STORAGE_SAS_TOKEN"],
)

repo = icechunk.Repository.create(storage)
session = repo.writable_session("main")

ds.to_zarr(
    session.store,
    mode="w",
    zarr_format=3,
    compute=False,
    write_empty_chunks=False,
    consolidated=False,
)

snapshot_id = session.commit("initialize store: empty template")
print(f"Store initialized. Snapshot ID: {snapshot_id}")